# Import library

In [1]:
import numpy as np
import cv2
import os
import time
import math
from PIL import Image
import matplotlib.pyplot as plt
import random
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader,Dataset
from torchvision import transforms
import torchvision
from torchsummary import summary

# Generator

In [2]:
class residual_block(nn.Module):
    def __init__(self):
        super(residual_block, self).__init__() 
        self.conv2d_layer = nn.Conv2d(in_channels=64,out_channels=64,kernel_size=(3,3),stride=(1,1),padding='same')
        self.batchnorm = nn.BatchNorm2d(64)
        self.activation_func = nn.PReLU()
        
    def forward(self,x):
        z = self.conv2d_layer(x)
        z = self.batchnorm(z)        
        z = self.activation_func(z)        
        z = self.conv2d_layer(z)        
        z = self.batchnorm(z)        
        return torch.add(z,x)  

In [3]:
class GenLastBlock(nn.Module):
    def __init__(self):
        super(GenLastBlock, self).__init__()
        self.conv_l = nn.Conv2d(in_channels=64,out_channels=256,kernel_size=(9,9),stride=(1,1),padding='same')
        # self.conv_2 = nn.Conv2d(in_channels=256,out_channels=256,kernel_size=(9,9),stride=(1,1),padding='same')
        self.pixel_shuffle = nn.PixelShuffle(2) 
        self.act_l = nn.PReLU(64)
    def forward(self,x):
        x = self.conv_l(x)       
        x = self.pixel_shuffle(x)        
        x = self.act_l(x)
        x = self.conv_l(x)
        x = self.pixel_shuffle(x)
        x = self.act_l(x)
        return x

In [4]:
class GeneratorClass(nn.Module):
    def __init__(self):
        super(GeneratorClass,self).__init__()
        self.res_block = residual_block()
        self.last_block = GenLastBlock()
        self.conv2d_l = nn.Conv2d(3,64,kernel_size=9,stride=1,padding='same')
        self.conv2d_2 = nn.Conv2d(in_channels=64,out_channels=64,kernel_size=(3,3),stride=(1,1),padding='same')
        self.batch_norm = nn.BatchNorm2d(64)
        self.act = nn.PReLU()
        self.last_conv2d = nn.Conv2d(in_channels=64,out_channels=3,kernel_size=(9,9),stride=(1,1),padding='same')
    def forward(self,input):
        x = self.conv2d_l(input)        
        x = self.act(x)
        temp = x        
        x = self.res_block(x)
        x = self.res_block(x)
        x = self.res_block(x)
        x = self.res_block(x)
        x = self.res_block(x)
        x = self.conv2d_2(x)
        x = self.batch_norm(x)
        x = torch.add(x,temp)
        x = self.last_block(x)
        x = self.last_conv2d(x)
        return x
        

# Discriminator

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator,self).__init__()
        self.input_conv = nn.Conv2d(in_channels=3,out_channels=64,kernel_size=(3,3),stride=(1,1),padding='same')
        self.act = nn.LeakyReLU(0.2)
        self.first_block = self.create_block(64,128,1)
        self.second_block = self.create_block(128,128,2)
        self.third_block = self.create_block(128,256,1)
        self.fourth_block = self.create_block(256,256,2)
        self.fifth_block = self.create_block(256,512,1)
        self.sixth_block = self.create_block(512,512,2)
        self.second_layer = nn.Linear(1179648,1024)
        self.adv_layer = nn.Sequential(nn.Linear(1024, 1), nn.Sigmoid())
          
    def create_block(self,in_num,out_num,stride_num):
        layer_block = nn.Sequential(
            nn.Conv2d(in_channels=in_num,out_channels=out_num,kernel_size=(3,3),stride=(stride_num,stride_num),padding=1),
            nn.BatchNorm2d(out_num),
            nn.LeakyReLU(0.2,inplace=True)
        )
        return layer_block
    
    def forward(self,x):
        x = self.input_conv(x)
        x = self.act(x)
        x = self.first_block(x)
        # print(x.shape)
        x = self.second_block(x)
        x = self.third_block(x)
        x = self.fourth_block(x)
        x = self.fifth_block(x)
        x = self.sixth_block(x)
        x = x.view(x.shape[0], -1)
        x = self.second_layer(x)
        x = self.act(x)
        x = self.adv_layer(x)
        return x
        

In [6]:
temp_net = GeneratorClass()

In [7]:
temp_input = torch.rand((1,3,96,96))

In [8]:
output = temp_net(temp_input)

In [13]:
temp_dis = Discriminator()

In [14]:
final_out = temp_dis(output)

In [ ]:
final_out.shape